# Worksheet 09 - Selection methods for generative and predictive models

By the end of this section, students will be able to:

- Give examples of questions that can be answered by generative models and others that can be answered by predictive models.
- Discuss how the research question being asked impacts the statistical modelling procedures.
- Explain how regularized methods, such as LASSO and ridge, can be used to estimate a predictive or a generative model.
- Distinguish the selection properties of LASSO and ridge penalties.
- Discuss why the model obtained directly from LASSO is not the most suitable model for generative modelling and how post-LASSO is one way to address this problem.
- Write a computer script to perform LASSO/ridge and use it to predict new outcomes.
- Write a computer script to perform post-LASSO and use it to estimate a generative model.
- Discuss post-inference problems (e.g., double dipping into the data set) and current practical solutions available to address these (e.g., data-splitting techniques).
- Write a computer script to apply currently available practical solutions to post inference problems.
- Discuss how the research question being asked impacts the communication of the results.

In [ ]:
# Loading packages
library(car)
library(tidyverse)
library(tidymodels)
library(glmnet)
library(modeldata)
library(mltools)
source("tests_worksheet_09.R")


## In worksheet_08: 

### Select a model using *stepwise* algorithms

- these are *greedy* algorithms 

- results depend on the order in which variables are selected 

- variables are either *in* (i.e., estimated coefficient different from zero) or *out* (i.e., estimated coefficient equal to zero)

## In this worksheet:

- can we use regularized methods to select a predictive model? and for a generative model?

- can we use the selected model to make inference about the population?

## Regularized Regression

We have learned that *shrinkage methods* can be used to build predictive and generative models. Some of these methods can also be used for variable selection.

- *LASSO penalty* can be used to select variables in a regression model by shrinking some estimated coefficients to exactly zero.
    - It can be used when there are more predictors than observations
    - It does not necessarily select all dummy variables associated with a categorical predictor as a group. A group LASSO penalty can be used to achieve group-wise selection.
      
- *Ridge penalty* generally does not perform variable selection because it shrinks estimated coefficients toward zero without setting them exactly to zero.
    - It can be used when there are more predictors than observations.
    - It is particularly useful for addressing multicollinearity (in fact, this was one of its original motivations).
    - Like LASSO, it penalizes individual coefficients rather than treating dummy variables associated with a categorical predictor as a group.

In this worksheet, we will use the Housing dataset to perform variable selection using LASSO and then fit a multiple linear regression (MLR) model for inference. In the tutorial, we will use a different dataset and ridge regression to build a predictive model. Note that both penalties can be used to build predictive and generative models, while LASSO can also be used for variable selection.

## Model Selection For Inference

**Recall** the [Ames `Housing` dataset](https://www.kaggle.com/c/home-data-for-ml-course/) you worked with in `worksheet_08`. Let's refresh our memory: it was compiled by Dean De Cock and has 79 input variables on different characteristics of residential houses in Ames, Iowa, USA that can be used to predict the property's final price, `SalePrice`. As in `worksheet_08`, we will focus our attention on 21 numerical input variables:

- `LotFrontage`: Linear $\text{ft}$ of street connected to the house.
- `LotArea`: Lot size in $\text{ft}^2$.
- `MasVnrArea`: Masonry veneer area in $\text{ft}^2$.
- `TotalBsmtSF`: Total $\text{ft}^2$ of basement area.
- `GrLivArea`: Above grade (ground) living area in $\text{ft}^2$.
- `BsmtFullBath`: Number of full bathrooms in basement.
- `BsmtHalfBath`: Number of half bathrooms in basement.
- `FullBath`: Number of full bathrooms above grade.
- `HalfBath`: Number of half bathrooms above grade.
- `BedroomAbvGr`: Number of bedrooms above grade (it does not include basement bedrooms).
- `KitchenAbvGr`: Number of kitchens above grade.
- `Fireplaces`: Number of fireplaces.
- `GarageArea`: Garage area in $\text{ft}^2$.
- `WoodDeckSF`: Wood deck area in $\text{ft}^2$.
- `OpenPorchSF`: Open porch area in $\text{ft}^2$.
- `EnclosedPorch`: Enclosed porch area in $\text{ft}^2$.
- `ScreenPorch`: Screen porch area in $\text{ft}^2$.

Let's start by loading the data set. 

In [ ]:
## Load the housing data set
housing_raw <- read_csv("data/Housing.csv", col_types = cols())

# Use `YearBuilt` and `YrSold` to create a variable `ageSold`
housing_raw$ageSold <- housing_raw$YrSold - housing_raw$YearBuilt


# Select subset of input variables
housing_raw <- 
  housing_raw %>%
  select(LotFrontage, LotArea, MasVnrArea, TotalBsmtSF, 
    GrLivArea, BsmtFullBath, BsmtHalfBath, FullBath, HalfBath, BedroomAbvGr, KitchenAbvGr, Fireplaces,
    GarageArea, WoodDeckSF, OpenPorchSF, EnclosedPorch, ScreenPorch, PoolArea, ageSold, SalePrice
  )

# Remove those rows containing `NA`s and some outliers
housing_raw <- 
    drop_na(housing_raw)  %>% 
    filter(LotArea < 20000)

str(housing_raw)

Our objective in this worksheet is to obtain a model for inference. We want to study how the property values are affected by their attributes. We want to be able to:

1. Interpret the parameters of the model;
2. Identify relevant attributes (covariates); 
3. Have a measure of uncertainty of our estimates.

**Question 1.0** 
<br> {points: 1}

Since we do not know which variables are important/relevant, we will need to conduct a variable selection technique. Let's start by splitting the data set into two sets: (1) the first part, with 60% of the rows, will be used to select a model; and (2) the second part will be for inference. 

> Note that the partition is similar to the training-test one used in prediction problems.

Your job is to randomly select 60% of the rows and store them in an object called `housing_selection`. Store the remaining rows in an object called `housing_inference`.

The `housing_inference` object is golden! It should not be touched before we select the variables. No peeking!!  

In [ ]:
set.seed(20211118) # Do not change this

# Housing_split <- ...(..., prop = ..., strata = SalePrice)
# housing_selection <- training(Housing_split)
# housing_inference <- testing(Housing_split)

# your code here
fail() # No Answer - remove if you provide an answer

head(housing_selection)

In [ ]:
test_1.0()

**Question 1.1** 
<br> {points: 1}

In this question, we will use LASSO to select variables and then fit a multiple linear regression (MLR) model using a separate subset of the data, with the goal of making inferences about the population.

Run LASSO on the `housing_selection` tibble and find the value `lambda` that provides the lowest cross-validation MSE (see the `cv.glmnet` function).

_Save the result in an object named `lasso_model`._

In [ ]:
set.seed(20211118) # do not change this

# lasso_model <-
#     cv.glmnet(x = ... %>% ...(-...)%>% as.matrix(), 
#               y = ..., 
#               alpha = ...)

# your code here
fail() # No Answer - remove if you provide an answer

lasso_model

In [ ]:
test_1.1()

**Question 1.2** 
<br> {points: 1}

We can visualize the estimated cross-validation MSE for each value of `lambda` using `lasso_model` and `plot()`.

The resulting plot shows the cross-validation MSE on the y-axis (error bars indicate variation across the folds) and the values of `lambda` on the bottom x-axis on a natural log scale. The top axis shows the number of nonzero estimated coefficients.

> **Recall:** Unlike Ridge regression, LASSO can shrink estimated coefficients exactly to zero. Therefore, the number of nonzero coefficients shown on the top axis can decrease as `lambda` increases.

Plot the cross-validation results and store the number of nonzero coefficients at the value of `lambda` with the lowest cross-validation MSE in an object called `lasso_nzero_min`.

*Fill in the parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# plot(...)
# lasso_nzero_min <- ...$nzero[which.min(...$cvm)]

# your code here
fail() # No Answer - remove if you provide an answer

lasso_nzero_min

In [ ]:
test_1.2()

**Question 1.3** 
<br> {points: 1}

Write code to extract the coefficients of the best LASSO model found in the `lasso_model`. By best, we mean the one with the smallest MSE. 

_Save the result in an object named `beta_lasso`._

In [ ]:
set.seed(20211118) # do not change this


# your code here
fail() # No Answer - remove if you provide an answer

beta_lasso

In [ ]:
test_1.3()

**Question 1.4** 
<br> {points: 1}

Extract the names of the covariates selected by LASSO in an object named `lasso_selected_covariates`.  

In [ ]:
#lasso_selected_covariates <- as_tibble(
        # as.matrix(...),
        # rownames='covariate') %>%
        # filter(covariate != '(Intercept)' & abs(s1) !=0) %>% 
        # pull(...)

# your code here
fail() # No Answer - remove if you provide an answer

lasso_selected_covariates

In [ ]:
test_1.4()

**Question 1.5** 
<br> {points: 1}

We expect that LASSO would remove highly correlated variables. However, LASSO can still fit a linear model on data sets with high levels of multicollinearity. Unfortunately, ordinary least squares cannot. To be on the safe side, let's check the variance inflation factor of the variables selected by LASSO. 

_Save the output in an object named `lasso_variables_vif`._

In [ ]:
#lasso_variables_vif <- 
#    vif(...)

# your code here
fail() # No Answer - remove if you provide an answer

lasso_variables_vif

In [ ]:
test_1.5()

**Question 1.6**
<br>{points: 1}

True or false?

The values in `lasso_variables_vif` do not indicate a concerning level of multicollinearity.

_Assign your answer to an object called `answer1.6`. Your answer should be either "true" or "false", surrounded by quotes._

In [ ]:
# answer1.6 <- ...

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_1.6()

**Question 1.7** 
<br> {points: 1}

Finally, let's use the covariates selected by LASSO and stored in `lasso_selected_covariates` to fit a linear model using ordinary least squares.

_Save the output in an object named `inference_model`._

In [ ]:
# your code here
fail() # No Answer - remove if you provide an answer

summary(inference_model)

In [ ]:
test_1.7()